# Experiment 1 Baseline Analysis

Use this notebook to inspect saved severity-baseline runs, compare subsets such as `top_only` and `top_priority`, and produce reusable summary plots for the early Experiment 1 phase.

In [ ]:
from pathlib import Path

DATA_ROOT = Path("/home/tmushuru/datasets/alopecia_public")
RESULT_ROOT = Path("/home/tmushuru/github-repos/alopecia-federated-prognosis/experiments/results/exp01")

RUNS = {
    "top_only_v1": {
        "summary": DATA_ROOT / "metadata" / "exp01_top_only_baseline_summary.json",
        "train_manifest": DATA_ROOT / "manifests" / "exp01_top_only_train.csv",
        "val_manifest": DATA_ROOT / "manifests" / "exp01_top_only_val.csv",
    },
    "top_priority_v1": {
        "summary": DATA_ROOT / "metadata" / "exp01_top_priority_baseline_summary.json",
        "train_manifest": DATA_ROOT / "manifests" / "exp01_top_priority_train.csv",
        "val_manifest": DATA_ROOT / "manifests" / "exp01_top_priority_val.csv",
    },
    "top_priority_v2": {
        "summary": DATA_ROOT / "metadata" / "exp01_top_priority_baseline_v2_summary.json",
        "train_manifest": DATA_ROOT / "manifests" / "exp01_top_priority_train.csv",
        "val_manifest": DATA_ROOT / "manifests" / "exp01_top_priority_val.csv",
    },
}

DATA_ROOT, RESULT_ROOT

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("ggplot")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)


In [ ]:
def load_summary(path: Path) -> dict | None:
    if not path.exists():
        print(f"Missing summary: {path}")
        return None
    return json.loads(path.read_text())


def load_manifest(path: Path) -> pd.DataFrame | None:
    if not path.exists():
        print(f"Missing manifest: {path}")
        return None
    return pd.read_csv(path)


def history_frame(summary: dict, run_name: str) -> pd.DataFrame:
    frame = pd.DataFrame(summary["history"])
    frame["run"] = run_name
    return frame


In [ ]:
run_summaries = {
    name: load_summary(info["summary"])
    for name, info in RUNS.items()
}

summary_rows = []
for name, summary in run_summaries.items():
    if summary is None:
        continue
    last_epoch = summary["history"][-1] if summary.get("history") else {}
    best_val = max((epoch["val_accuracy"] for epoch in summary.get("history", [])), default=None)
    summary_rows.append(
        {
            "run": name,
            "device": summary.get("device"),
            "pretrained": summary.get("pretrained"),
            "augment": summary.get("augment"),
            "class_weighting": summary.get("class_weighting"),
            "train_samples": summary.get("train_sample_count"),
            "val_samples": summary.get("val_sample_count"),
            "final_train_accuracy": last_epoch.get("train_accuracy"),
            "final_val_accuracy": last_epoch.get("val_accuracy"),
            "best_val_accuracy": best_val,
            "final_train_loss": last_epoch.get("train_loss"),
            "final_val_loss": last_epoch.get("val_loss"),
        }
    )

pd.DataFrame(summary_rows).sort_values("run")

In [ ]:
history_frames = [
    history_frame(summary, name)
    for name, summary in run_summaries.items()
    if summary is not None and summary.get("history")
]

history_df = pd.concat(history_frames, ignore_index=True) if history_frames else pd.DataFrame()
history_df

In [ ]:
if not history_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

    for run_name, run_frame in history_df.groupby("run"):
        axes[0].plot(run_frame["epoch"], run_frame["train_loss"], marker="o", label=f"{run_name} train")
        axes[0].plot(run_frame["epoch"], run_frame["val_loss"], marker="x", linestyle="--", label=f"{run_name} val")
        axes[1].plot(run_frame["epoch"], run_frame["train_accuracy"], marker="o", label=f"{run_name} train")
        axes[1].plot(run_frame["epoch"], run_frame["val_accuracy"], marker="x", linestyle="--", label=f"{run_name} val")

    axes[0].set_title("Loss Curves")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend(fontsize=8)

    axes[1].set_title("Accuracy Curves")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend(fontsize=8)

    plt.tight_layout()
else:
    print("No histories available to plot yet.")

In [ ]:
manifest_frames = {}
for run_name, info in RUNS.items():
    train_df = load_manifest(info["train_manifest"])
    val_df = load_manifest(info["val_manifest"])
    if train_df is not None:
        train_df = train_df.assign(split_source="train", run=run_name)
    if val_df is not None:
        val_df = val_df.assign(split_source="val", run=run_name)
    frames = [frame for frame in [train_df, val_df] if frame is not None]
    if frames:
        manifest_frames[run_name] = pd.concat(frames, ignore_index=True)

{name: frame.shape for name, frame in manifest_frames.items()}

In [ ]:
for run_name, frame in manifest_frames.items():
    print(f"\n=== {run_name} ===")
    print("Dataset counts:")
    print(frame["dataset_key"].value_counts())
    print("\nView counts:")
    print(frame["image_view"].value_counts())
    print("\nSeverity counts:")
    print(frame["severity_proxy_value"].value_counts().sort_index())

In [ ]:
if manifest_frames:
    fig, axes = plt.subplots(1, len(manifest_frames), figsize=(5 * len(manifest_frames), 4), squeeze=False)
    axes = axes[0]
    for axis, (run_name, frame) in zip(axes, manifest_frames.items()):
        severity_counts = frame["severity_proxy_value"].value_counts().sort_index()
        axis.bar(severity_counts.index.astype(str), severity_counts.values)
        axis.set_title(f"{run_name} severity distribution")
        axis.set_xlabel("Severity")
        axis.set_ylabel("Count")
    plt.tight_layout()
else:
    print("No manifest data available to plot yet.")

## Notes

- `top_only` is useful for clean loader and smoke-test checks.
- `top_priority` is the current working subset for early baseline development.
- Keep saving machine-readable JSON summaries first, then use this notebook to compare runs over time.